# CMC 87708 — Gemma four-condition characterization (768)

Completes the revised-manuscript four-condition Gemma analysis:
- LLM-only control
- Flow metadata
- Graph context
- Detector-state condition

The same 500 held-out targets per dataset are reused. The four conditions are characterized within role; they are not treated as a common performance ladder.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

!git clone https://github.com/ramita99/cmc_llm_87708.git
%cd cmc_llm_87708
!pip install -q -e .

In [ ]:
import json, yaml, subprocess, sys, shutil
from pathlib import Path
import pandas as pd
import torch
from google.colab import userdata
from huggingface_hub import login, snapshot_download

hf_token=userdata.get("HF_TOKEN")
if hf_token:
    login(token=hf_token,add_to_git_credential=False)

SOURCE_ROOT=Path("/content/drive/MyDrive/CMC_87708_primary_packets")
PRIMARY_GEMMA_ROOT=Path("/content/drive/MyDrive/CMC_87708_primary_gemma")
THREE_ROOT=Path("/content/drive/MyDrive/CMC_87708_gemma_three_state")
COMBINED_ROOT=Path("/content/drive/MyDrive/CMC_87708_gemma_four_condition")
DATASETS=["NF-UNSW-NB15-v3","NF-CSE-CIC-IDS2018-v3","NF-ToN-IoT-v3"]
THREE_STATES=["llm_only","flow","graph"]

def load_jsonl(p):
    with Path(p).open("r",encoding="utf-8") as f:
        return [json.loads(x) for x in f if x.strip()]

for sub in ["packets","manifests","results"]:
    (THREE_ROOT/sub).mkdir(parents=True,exist_ok=True)

for ds in DATASETS:
    packets=load_jsonl(SOURCE_ROOT/"packets"/f"{ds}.jsonl")
    selected=[p for p in packets if p.get("state") in THREE_STATES]
    counts=pd.Series([p["state"] for p in selected]).value_counts().to_dict()
    assert counts=={"llm_only":500,"flow":500,"graph":500},(ds,counts)
    with (THREE_ROOT/"packets"/f"{ds}.jsonl").open("w",encoding="utf-8") as f:
        for p in selected:
            f.write(json.dumps(p,ensure_ascii=False,default=str)+"\n")

In [ ]:
MODEL_ID="google/gemma-2-9b-it"
MODEL_REVISION="11c9b309abf73637e4b6f9a3fa1e92e615547819"
LOCAL_MODEL=snapshot_download(repo_id=MODEL_ID,revision=MODEL_REVISION,token=hf_token)

cfg={
    "protocol_version":"CMC-87708-GEMMA-THREE-STATE-768",
    "output_root":str(THREE_ROOT),
    "sample_seed":7,
    "targets_per_dataset":500,
    "states":THREE_STATES,
    "methods":["proposed_source_gated"],
    "full_state_for_method_comparison":"detector",
    "generation_dtype":"bfloat16",
    "attention_implementation":"sdpa",
    "generation_batch_size":4,
    "max_new_tokens":768,
    "do_sample":False,
    "numeric_tolerance":0.0001,
    "fallback_on_truncation":True,
    "bootstrap_replicates":10000,
}
CFG=Path("/content/gemma_three_state_768.yaml")
CFG.write_text(yaml.safe_dump(cfg,sort_keys=False),encoding="utf-8")

subprocess.run([
    sys.executable,"scripts/02_run_primary_generation.py",
    "--experiment",str(CFG),"--model",str(LOCAL_MODEL),
    "--methods","proposed_source_gated","--resume"
],check=True)

In [ ]:
# Merge the three newly generated states with the Gemma primary Detector-state Source-gated outputs.
for sub in ["packets","outputs/proposed_source_gated","results","manifests"]:
    (COMBINED_ROOT/sub).mkdir(parents=True,exist_ok=True)

for ds in DATASETS:
    source_packets=load_jsonl(SOURCE_ROOT/"packets"/f"{ds}.jsonl")
    assert len(source_packets)==2000
    with (COMBINED_ROOT/"packets"/f"{ds}.jsonl").open("w",encoding="utf-8") as f:
        for p in source_packets:
            f.write(json.dumps(p,ensure_ascii=False,default=str)+"\n")

    three=load_jsonl(THREE_ROOT/"outputs"/"proposed_source_gated"/f"{ds}.jsonl")
    detector=load_jsonl(PRIMARY_GEMMA_ROOT/"outputs"/"proposed_source_gated"/f"{ds}.jsonl")
    assert len(three)==1500 and len(detector)==500
    merged=three+detector
    assert pd.Series([r["state"] for r in merged]).value_counts().to_dict()=={
        "llm_only":500,"flow":500,"graph":500,"detector":500
    }
    with (COMBINED_ROOT/"outputs"/"proposed_source_gated"/f"{ds}.jsonl").open("w",encoding="utf-8") as f:
        for r in merged:
            f.write(json.dumps(r,ensure_ascii=False,default=str)+"\n")

combined_cfg=dict(cfg)
combined_cfg.update({
    "protocol_version":"CMC-87708-GEMMA-FOUR-CONDITION-768",
    "output_root":str(COMBINED_ROOT),
    "states":["llm_only","flow","graph","detector"],
})
COMBINED_CFG=Path("/content/gemma_four_condition_768.yaml")
COMBINED_CFG.write_text(yaml.safe_dump(combined_cfg,sort_keys=False),encoding="utf-8")
subprocess.run([sys.executable,"scripts/03_recompute_primary_metrics.py","--experiment",str(COMBINED_CFG)],check=True)
print("Gemma four-condition characterization complete.")